# Unstructured data

## PDF

Problem:

My company wants to buy a graphic cards to create an AI cluster. Do a market research so we know what to choose.

https://www.nvidia.com/en-us/design-visualization/desktop-graphics/

In [ ]:
# !pip3 install PyMuPDF
import fitz
import re

In [ ]:
# Open file
filepath = '../data/NVIDIA RTX A4000 Datasheet.pdf'
pages = []
with fitz.open(filepath) as doc:
    for page in doc:
        text = page.get_text()
        pages.append(text)

In [ ]:
text = pages[0]
text = text.replace('\n','')
text = text.replace(',','')

In [ ]:
memory = re.findall(r'GPU memory(.*)GB GDDR6Memory interface', text)[0]
cuda_cores = re.findall(r'CUDA Cores(.*)NVIDIA third-generation', text)[0]
tensor_cores = re.findall(r'NVIDIA third-generation Tensor Cores(.*)NVIDIA second-generation', text)[0]
rt_cores = re.findall(r'NVIDIA second-generation RT Cores(.*)Single-precision', text)[0]

memory, cuda_cores, tensor_cores, rt_cores

In [ ]:
from pathlib import Path

In [ ]:
# Convert it to a function

def get_catalog_attributes(filepath):

    filename = Path(filepath).stem

    pages = []
    with fitz.open(filepath) as doc:
        for page in doc:
            text = page.get_text()
            pages.append(text)

    text = pages[0]
    text = text.replace('\n','')
    text = text.replace(',','')

    memory = re.findall(r'GPU memory(.*)GB GDDR6Memory interface', text)[0]
    cuda_cores = re.findall(r'CUDA Cores(.*)NVIDIA third-generation', text)[0]
    tensor_cores = re.findall(r'NVIDIA third-generation Tensor Cores(.*)NVIDIA second-generation', text)[0]
    rt_cores = re.findall(r'NVIDIA second-generation RT Cores(.*)Single-precision', text)[0]

    output = {filename: {'memory':memory, 'cuda_cores':cuda_cores, 'tensor_cores':tensor_cores, 'rt_cores':rt_cores}}

    return output

In [ ]:
get_catalog_attributes('../data/NVIDIA RTX A4000 Datasheet.pdf')

In [ ]:
def get_catalog_attributes(filepaths):

    filenames = []
    memory = []
    cuda_cores = []
    tensor_cores = []
    rt_cores = []
    power = []

    for filepath in filepaths:
        filename = Path(filepath).stem

        pages = []
        with fitz.open(filepath) as doc:
            for page in doc:
                text = page.get_text()
                pages.append(text)

        text = pages[0]
        text = text.replace('\n','')
        text = text.replace(',','')
        text = text.replace(':','')
        text = text.replace('  ',' ')

        filenames.append(filename)
        memory.append(re.findall(r'GPU memory(.*)GB GDDR6Memory interface', text)[0])
        cuda_cores.append(re.findall(r'CUDA Cores(.*)NVIDIA third-generation', text)[0])
        tensor_cores.append(re.findall(r'NVIDIA third-generation Tensor Cores(.*)NVIDIA second-generation', text)[0])
        rt_cores.append(re.findall(r'NVIDIA second-generation RT Cores(.*)Single-precision', text)[0])
        power.append(re.findall(r'Total board power(.*) WThermal', text)[0])

    output = {'file':filenames, 'memory':memory, 'cuda_cores':cuda_cores, 'tensor_cores':tensor_cores, 'rt_cores':rt_cores, 'power':power}

    return output

In [ ]:
filepaths = [
    '../data/NVIDIA RTX A2000 Datasheet.pdf',
    '../data/NVIDIA RTX A4000 Datasheet.pdf',
    '../data/NVIDIA RTX A4500 Datasheet.pdf',
    '../data/NVIDIA RTX A5000 Datasheet.pdf',
    '../data/NVIDIA RTX A5500 Datasheet.pdf',
    '../data/NVIDIA RTX A6000 Datasheet.pdf']
out = get_catalog_attributes(filepaths)
out

In [ ]:
import pandas as pd
df = pd.DataFrame(out)
df.file = df.file.replace('NVIDIA RTX ', '', regex=True)
df.file = df.file.replace(' Datasheet', '', regex=True)
df

In [ ]:
for col in df.columns:
    try:
        df[col] = df[col].astype('int')
    except Exception as e:
        print(e, col)

In [ ]:
df.dtypes

In [ ]:
df.cuda_cores.plot()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20,5))
plt.bar(x = df.file, height = df.cuda_cores)

In [ ]:
# Ideally should be done with webscraping, but for now is fine with a simple search
df['price'] = [769, 999, 1299, 1699, 2251, 4129]

In [ ]:
import seaborn as sns
sns.pairplot(data=df)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df.power, df.cuda_cores, c=df.price)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
plt.scatter(df.cuda_cores, df.price, c=df.power)

for i, txt in enumerate(df.file):
    ax.annotate(txt, (df.cuda_cores[i], df.price[i]))

# Docx

In [ ]:
# pip3 install python-docx
from docx import Document

In [ ]:
speech = Document('../data/Steve Jobs Stanford Speech.docx')
speech

In [ ]:
# help(Document)

In [ ]:
texts = []
for paragraph in speech.paragraphs:
    texts.append(paragraph.text)
    # print(paragraph.text)

In [ ]:
def keep_letters(s):
    return ''.join([i for i in s if i.isalpha() or i == ' '])

In [ ]:
text = " ".join(texts)
text = keep_letters(text)
text = text.lower()
text = text.replace('  ', ' ')
text

In [ ]:
words = []
num = []
for word in set(text.split(' ')):
    words.append(word)
    num.append(text.count(word))
    # print(word, text.count(word))

In [ ]:
df = pd.DataFrame({'word':words, 'num':num})

df = df.loc[df.word.str.len() >= 5]
df.sort_values(by='num', ascending=False).head(20)

## OCR

Get text from a simple image.

### 1 Install tesseract

Installation instructions: <https://tesseract-ocr.github.io/tessdoc/Installation.html>

- Ubuntu/debian

```shell
sudo apt install tesseract-ocr
sudo apt install libtesseract-dev
```

- Windows

<https://github.com/UB-Mannheim/tesseract/wiki>

```python
# If you don't have tesseract executable in your PATH, include the following:
pytesseract.pytesseract.tesseract_cmd = r'C:\ProgramFiles\Tesseract-OCR\tesseract'
```

### 2 Install python wrapper

```shell
poetry add pytesseract
```



In [ ]:
import pytesseract

In [ ]:
image = '../data/image.jpg'

In [ ]:
ocr = pytesseract.image_to_string(image)
print(ocr)

Convert a pdf to text

Source: https://vault.fbi.gov/soviet-active-measures-relating-to-the-u.s.-peace-movement/soviet-active-measures-relating-to-the-u.s.-peace-movement-part-01/view

In [ ]:
pdffile = '../data/Soviet Active Measures Relating to the US Peace Movement Part 01.pdf'
doc = fitz.open(pdffile)

In [ ]:
from PIL import Image
import io

In [ ]:
doc = fitz.open(pdffile)
page = doc.load_page(5)  # number of page
pix = page.get_pixmap(dpi=300)

In [ ]:
pix.save('../data/test.jpg')
# We would reduce HDD usage if we save directly to memory, using io
img = Image.open('../data/test.jpg')
text = pytesseract.image_to_string(img)
text

In [ ]:
def pdf_to_text(pdffile, dpi=300, lang='eng'):
    """ Converts a pdf page to text

    Args:
        pdffile  (pathlib.Path): Path of pdf file
        dpi      (int)         : Defaults to 300.
        lang     (str)         : Defaults to 'eng'. Other options: 'spa', 'equ'

    Returns:
        text      (str)         : Parsed text of the page
    """
    doc = fitz.open(pdffile)

    pages = []
    # Use "png" as the correct image format for pix.tobytes
    for num_page in range(doc.page_count):
        page = doc.load_page(num_page)  # number of page
        pix = page.get_pixmap(dpi=dpi)

        data = pix.tobytes("png")  # Use png, a supported format
        # Save img to io
        img = Image.open(io.BytesIO(data))
        text = pytesseract.image_to_string(img, lang=lang)
        pages.append(text)
        print(text)
    return pages

In [ ]:
pdf_to_text(pdffile)